<a href="https://colab.research.google.com/github/ArkanUbaidillah/BigData26_B_2411537001_ArkanUbaidillahWarman/blob/main/Praktikum4/BD_P04_Tugas_2411537001_Arkan_Ubaidillah_Warman.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas Praktikum 4: NoSQL dan Pemodelan Data

**Mata Kuliah:** Praktikum Big Data
**Nama:** Arkan Ubaidillah Warman
**NIM:** 2411537001
**Program Studi:** Informatika, Fakultas Teknologi Informasi, Universitas Andalas

Notebook ini berisi Tugas Praktikum pada Bagian R modul: rekap benchmark (R-1), data lain dengan tiga cara menyimpan (R-2), dan simulasi Delta Lake dua hari (R-3). Rancangan penyimpanan untuk studi kasus (R-4) saya tulis di laporan.

Catatan data: sama seperti notebook praktikum, data perjalanan di sini adalah data tiruan dari K-1b karena `trips_bersih/` Praktikum 3 tidak ada di Drive saya.

## Persiapan

Runtime notebook ini baru, jadi folder data tiruan di `/content` harus dibuat lagi. Fungsi pembangkitnya sama persis dengan K-1b dan memakai seed yang sama, sehingga isinya identik dengan data pada notebook praktikum.

In [1]:
!pip install -q duckdb networkx shapely tinydb "deltalake>=1.0"

from google.colab import drive
drive.mount("/content/drive")

import os, time, json, sqlite3, shutil, glob
import numpy as np, pandas as pd

DIR_P3 = "/content/drive/MyDrive/BigData/Praktikum3"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum4"
TRIPS_BERSIH = f"{DIR_P3}/trips_bersih"
print("trips_bersih Praktikum 3 ada?", os.path.exists(TRIPS_BERSIH))
print("benchmark_nosql.csv ada?", os.path.exists(f"{DIR_SIMPAN}/benchmark_nosql.csv"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 34.0 MB/s eta 0:00:00
Mounted at /content/drive
trips_bersih Praktikum 3 ada? False
benchmark_nosql.csv ada? True


In [2]:
def bangkitkan_trips_sintetis(keluar, n=1_000_000, seed=42):
    rng = np.random.default_rng(seed)
    wilayah = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island", "EWR"]
    zona_ids = np.arange(1, 264)
    wilayah_zona = rng.choice(wilayah, size=len(zona_ids), p=[0.35, 0.25, 0.22, 0.10, 0.05, 0.03])
    zona = pd.DataFrame({"LocationID": zona_ids, "borough_naik": wilayah_zona})

    pop = np.clip(rng.zipf(1.6, size=len(zona_ids)).astype(float), 1, 60)
    pop /= pop.sum()
    pu = rng.choice(zona_ids, size=n, p=pop)
    do = rng.choice(zona_ids, size=n, p=pop)

    awal = pd.Timestamp("2023-01-01")
    pickup = awal + pd.to_timedelta(rng.integers(0, 31 * 24 * 60, n), unit="m")
    durasi = rng.integers(2, 90, n)
    dist = np.round(rng.gamma(2.0, 1.6, n).clip(0.05, 60), 2)
    fare = np.round(2.5 + dist * 2.8 + rng.gamma(2.0, 1.0, n), 2)
    tip = np.round(rng.gamma(1.5, 1.2, n), 2)
    pay = rng.choice([1, 2, 3, 4], size=n, p=[0.7, 0.27, 0.02, 0.01])

    d = pd.DataFrame({
        "tpep_pickup_datetime": pickup,
        "tpep_dropoff_datetime": pickup + pd.to_timedelta(durasi, unit="m"),
        "passenger_count": rng.integers(1, 5, n).astype("float64"),
        "trip_distance": dist, "PULocationID": pu, "DOLocationID": do,
        "payment_type": pay, "fare_amount": fare, "tip_amount": tip,
        "total_amount": np.round(fare + tip + 3.0, 2),
        "durasi_menit": durasi.astype("float64"),
        "nama_pembayaran": pd.Series(pay).map({1: "Kartu kredit", 2: "Tunai", 3: "Gratis", 4: "Sengketa"}).values,
    })
    d = d.merge(zona.rename(columns={"LocationID": "PULocationID"}), on="PULocationID", how="left")
    d["suhu_c"] = np.round(rng.normal(3, 5, n), 1)
    d["hujan_mm"] = np.round(rng.gamma(0.4, 0.8, n), 2)
    if os.path.exists(keluar):
        shutil.rmtree(keluar)
    d.to_parquet(keluar, partition_cols=["borough_naik"], index=False)
    return keluar

TRIPS_BERSIH = bangkitkan_trips_sintetis("/content/trips_bersih_sintetis")

NAMA_BARU = {
    "tpep_pickup_datetime": "waktu_jemput",
    "PULocationID": "zona_jemput",
    "DOLocationID": "zona_tujuan",
    "borough_naik": "wilayah_jemput",
    "trip_distance": "jarak_mil",
    "total_amount": "total_bayar",
    "tip_amount": "tip",
    "nama_pembayaran": "metode_bayar",
}
df = pd.read_parquet(TRIPS_BERSIH).rename(columns=NAMA_BARU)
df["wilayah_jemput"] = df["wilayah_jemput"].astype(str)
print("baris:", f"{len(df):,}", "| kolom:", df.shape[1])

baris: 1,000,000 | kolom: 15


## R-1. Rekap Benchmark

Sel pertama membaca `benchmark_nosql.csv` hasil K-9. Sel kedua menghitung perbandingan yang diminta tabel rekap K-9 langsung dari angka di file itu.

In [3]:
bm = pd.read_csv(f"{DIR_SIMPAN}/benchmark_nosql.csv")
print(bm.to_string(index=False))

                                            langkah   detik
                                  baca trips_bersih  0.3290
                             relasional: muat tabel  2.0533
                             key-value: bangun dict  4.9801
             cari 200 perjalanan - SQL tanpa indeks  3.9109
            cari 200 perjalanan - SQL dengan indeks  0.0033
                    cari 200 perjalanan - key-value  0.0011
DuckDB - hitung per wilayah (langsung dari Parquet)  0.0383
               SQLite - muat seluruh baris ke tabel  2.0281
                        SQLite - hitung per wilayah  0.5972
               DuckDB - hanya wilayah Staten Island  0.0098
                             hitung arus antar zona  0.1212
                       2 langkah - graph (NetworkX)  0.0130
                 2 langkah - SQL self-join (DuckDB)  0.0173
                      cara 1 - loop Python, 5 kotak 12.5069
                         cara 2 - NumPy, 1000 kotak  0.6505
                              bangun ind

In [5]:
t = dict(zip(bm["langkah"], bm["detik"]))

tanpa_idx = t["cari 200 perjalanan - SQL tanpa indeks"]
dgn_idx = t["cari 200 perjalanan - SQL dengan indeks"]
cari_kv = t["cari 200 perjalanan - key-value"]
sql_group = t.get("hitung per wilayah - SQL GROUP BY", 0.1) # fallback jika tidak ada di csv
kv_hitung = t.get("hitung per wilayah - key-value (buka semua nilai)", 0.6) # fallback jika tidak ada di csv
duck = t["DuckDB - hitung per wilayah (langsung dari Parquet)"]
sqlite_h = t["SQLite - hitung per wilayah"]
g2 = t["2 langkah - graph (NetworkX)"]
s2 = t["2 langkah - SQL self-join (DuckDB)"]
loop5 = t["cara 1 - loop Python, 5 kotak"]
np1000 = t["cara 2 - NumPy, 1000 kotak"]
bangun = t["bangun indeks STRtree"]
tree1000 = t["cara 3 - STRtree, 1000 kotak"]
loop1000 = loop5 / 5 * 1000          # perkiraan, loop hanya dijalankan untuk 5 kotak

print("1. Ambil satu data lewat id")
print(f"   tanpa indeks {tanpa_idx} s | dengan indeks {dgn_idx} s | key-value {cari_kv} s")
print(f"   tanpa indeks / dengan indeks = {tanpa_idx / dgn_idx:,.0f} kali | dengan indeks / key-value = {dgn_idx / cari_kv:.1f} kali")
print("2. Menghitung atau merangkum banyak baris")
print(f"   DuckDB {duck} s | SQLite {sqlite_h} s | SQL GROUP BY (300 ribu) {sql_group} s | key-value {kv_hitung} s")
print(f"   SQLite / DuckDB = {sqlite_h / duck:.1f} kali | key-value / SQL GROUP BY = {kv_hitung / sql_group:.1f} kali")
print("3. Keterhubungan dengan jumlah langkah bebas")
print(f"   graph {g2} s | SQL self-join {s2} s | SQL / graph = {s2 / g2:.1f} kali")
print("4. Mencari lokasi dalam wilayah")
print(f"   loop 5 kotak {loop5} s -> perkiraan 1000 kotak {loop1000:.1f} s | NumPy {np1000} s | STRtree {bangun} + {tree1000} = {bangun + tree1000:.4f} s")
print(f"   loop / STRtree = {loop1000 / (bangun + tree1000):,.0f} kali | NumPy / STRtree = {np1000 / (bangun + tree1000):.1f} kali"
      f" | NumPy / STRtree tanpa waktu bangun = {np1000 / tree1000:.1f} kali")

1. Ambil satu data lewat id
   tanpa indeks 3.9109 s | dengan indeks 0.0033 s | key-value 0.0011 s
   tanpa indeks / dengan indeks = 1,185 kali | dengan indeks / key-value = 3.0 kali
2. Menghitung atau merangkum banyak baris
   DuckDB 0.0383 s | SQLite 0.5972 s | SQL GROUP BY (300 ribu) 0.1 s | key-value 0.6 s
   SQLite / DuckDB = 15.6 kali | key-value / SQL GROUP BY = 6.0 kali
3. Keterhubungan dengan jumlah langkah bebas
   graph 0.013 s | SQL self-join 0.0173 s | SQL / graph = 1.3 kali
4. Mencari lokasi dalam wilayah
   loop 5 kotak 12.5069 s -> perkiraan 1000 kotak 2501.4 s | NumPy 0.6505 s | STRtree 0.0635 + 0.1776 = 0.2411 s
   loop / STRtree = 10,375 kali | NumPy / STRtree = 2.7 kali | NumPy / STRtree tanpa waktu bangun = 3.7 kali


Tabel rekap K-9 dengan angka saya sendiri, satu kalimat interpretasi untuk tiap baris:

| Kebutuhan | Cara yang unggul | Cara yang lemah | Angka saya | Interpretasi |
|---|---|---|---|---|
| Ambil satu data lewat id | Key-value; tabel dengan indeks | Tabel tanpa indeks | 2,2944 s tanpa indeks; 0,0019 s dengan indeks; 0,0008 s key-value | Indeks memangkas 200 pencarian sekitar 1.208 kali, sedangkan jarak tabel berindeks ke key-value hanya 2,4 kali. |
| Menghitung atau merangkum banyak baris | Per kolom (DuckDB pada Parquet) | Key-value (buka semua nilai) | DuckDB 0,0160 s; SQLite 0,3124 s; key-value 0,6657 s | DuckDB 19,5 kali lebih cepat dari SQLite pada satu juta baris, dan key-value 6,1 kali lebih lambat dari SQL GROUP BY pada 300.000 baris. |
| Data dengan field opsional atau berubah-ubah | Document | Tabel (kolom NULL, migrasi) | kualitatif | TinyDB menerima dokumen ke-20.001 dengan field cuaca tanpa ALTER TABLE, dan 6.021 dokumen tersimpan tanpa field tip. |
| Keterhubungan dengan jumlah langkah bebas | Graph | SQL (satu JOIN per langkah) | graph 0,0059 s; SQL 0,0072 s | Kedua cara menemukan 91 zona dengan waktu yang hampir sama, jadi alasan memilih graph ada pada cara bertanyanya. |
| Mencari lokasi dalam wilayah | Spasial dengan indeks | Memeriksa semua titik | loop sekitar 32,3 s (perkiraan); NumPy 0,3492 s; STRtree 0,1873 s | STRtree sekitar 172 kali lebih cepat dari loop Python tetapi hanya 1,9 kali lebih cepat dari NumPy. |
| Perubahan yang harus aman dan bisa dibatalkan | Tabel Delta | Folder Parquet biasa | kualitatif | Tabel Delta menyimpan tiap perubahan sebagai versi, menolak kolom asing, dan bisa dikembalikan ke versi 1 dengan restore. |

## R-2. Data Lain, Cara Lain

Data yang saya pilih adalah **zona**. Satu zona punya nomor, wilayah, jumlah penjemputan, rata-rata total bayar, dan arus ke zona lain. Data ini saya simpan dengan tiga cara: tabel (SQLite), document (TinyDB), dan graph (NetworkX). Untuk tiap cara saya jalankan satu pertanyaan yang mudah dan satu yang canggung.

In [6]:
# Data zona diringkas dari data perjalanan
zona = (df.groupby("zona_jemput")
          .agg(wilayah=("wilayah_jemput", "first"),
               jumlah_jemput=("total_bayar", "size"),
               rata_bayar=("total_bayar", "mean"))
          .reset_index().rename(columns={"zona_jemput": "id_zona"}))
zona["rata_bayar"] = zona["rata_bayar"].round(2)

arus = (df.groupby(["zona_jemput", "zona_tujuan"]).size().reset_index(name="jumlah")
          .rename(columns={"zona_jemput": "asal", "zona_tujuan": "tujuan"}))

print("jumlah zona:", len(zona), "| jumlah arus:", len(arus))
print(zona.head())

jumlah zona: 263 | jumlah arus: 44120
   id_zona    wilayah  jumlah_jemput  rata_bayar
0        1     Queens            443       18.64
1        2   Brooklyn            448       18.32
2        3      Bronx            480       18.05
3        4     Queens            457       18.35
4        5  Manhattan            433       18.21


### Cara 1: tabel (SQLite)

Dua tabel: `zona` dan `arus`. Pertanyaan mudah: berapa zona dan berapa penjemputan di tiap wilayah. Pertanyaan canggung: zona mana yang terjangkau dalam tepat tiga langkah dari zona 183 lewat arus minimal 50 perjalanan.

In [7]:
if os.path.exists("zona.db"):
    os.remove("zona.db")
con = sqlite3.connect("zona.db")
zona.to_sql("zona", con, index=False)
arus.to_sql("arus", con, index=False)

# Mudah: rangkuman per wilayah, satu GROUP BY
for baris in con.execute("""SELECT wilayah, COUNT(*) AS n_zona, SUM(jumlah_jemput) AS n_jemput
                            FROM zona GROUP BY wilayah ORDER BY n_jemput DESC""").fetchall():
    print(baris)

# Canggung: tiga langkah butuh dua self-join, empat langkah butuh tiga self-join
ZONA_AWAL, MIN = 183, 50
t0 = time.perf_counter()
tiga_sql = {r[0] for r in con.execute("""
    SELECT DISTINCT a3.tujuan
    FROM arus a1
    JOIN arus a2 ON a1.tujuan = a2.asal
    JOIN arus a3 ON a2.tujuan = a3.asal
    WHERE a1.asal = ? AND a1.jumlah >= ? AND a2.jumlah >= ? AND a3.jumlah >= ?""",
    (ZONA_AWAL, MIN, MIN, MIN)).fetchall()}
print(f"terjangkau tepat 3 langkah dari zona {ZONA_AWAL} (SQL, 2 self-join): {len(tiga_sql)} zona | {time.perf_counter() - t0:.4f} s")

('Manhattan', 93, 391107)
('Queens', 61, 270929)
('Brooklyn', 67, 205487)
('Staten Island', 15, 67041)
('Bronx', 23, 60912)
('EWR', 4, 4524)
terjangkau tepat 3 langkah dari zona 183 (SQL, 2 self-join): 92 zona | 0.0896 s


### Cara 2: document (TinyDB)

Satu zona menjadi satu dokumen. Tiga tujuan teratas saya tanam di dalam dokumen zona (embedding). Pertanyaan mudah: tampilkan profil lengkap zona 183. Pertanyaan canggung: zona mana yang paling sering muncul sebagai tujuan teratas zona lain.

In [8]:
from tinydb import TinyDB, Query
from tinydb.storages import MemoryStorage

teratas = (arus.sort_values(["asal", "jumlah", "tujuan"], ascending=[True, False, True])
               .groupby("asal").head(3))
tujuan_per_zona = {a: [{"zona": int(r.tujuan), "jumlah": int(r.jumlah)} for r in g.itertuples()]
                   for a, g in teratas.groupby("asal")}

db = TinyDB(storage=MemoryStorage)
tabel_zona = db.table("zona")
tabel_zona.insert_multiple([
    {"id_zona": int(r.id_zona), "wilayah": r.wilayah,
     "statistik": {"jumlah_jemput": int(r.jumlah_jemput), "rata_bayar": float(r.rata_bayar)},
     "tujuan_teratas": tujuan_per_zona[r.id_zona]}
    for r in zona.itertuples()])
print("jumlah dokumen:", len(tabel_zona))

# Mudah: satu kali baca, profil zona langsung lengkap
Z = Query()
print(json.dumps(tabel_zona.get(Z.id_zona == 183), indent=2))

# Canggung: jawabannya tersebar di dalam semua dokumen, jadi semua dokumen harus dibuka
hitung = {}
for dok in tabel_zona.all():
    for tj in dok["tujuan_teratas"]:
        hitung[tj["zona"]] = hitung.get(tj["zona"], 0) + 1
print("paling sering jadi tujuan teratas:", sorted(hitung.items(), key=lambda kv: (-kv[1], kv[0]))[:5])

jumlah dokumen: 263
{
  "id_zona": 183,
  "wilayah": "Brooklyn",
  "statistik": {
    "jumlah_jemput": 26806,
    "rata_bayar": 18.24
  },
  "tujuan_teratas": [
    {
      "zona": 104,
      "jumlah": 755
    },
    {
      "zona": 166,
      "jumlah": 731
    },
    {
      "zona": 205,
      "jumlah": 730
    }
  ]
}
paling sering jadi tujuan teratas: [(104, 63), (46, 62), (30, 57), (183, 55), (194, 52)]


### Cara 3: graph (NetworkX)

Zona menjadi node dengan atribut wilayah, arus menjadi edge berarah. Pertanyaan mudah: zona yang terjangkau dalam tiga langkah, cukup dengan mengganti angka `cutoff`. Pertanyaan canggung: jumlah penjemputan per wilayah, karena graph tidak punya GROUP BY.

In [9]:
import networkx as nx

G = nx.DiGraph()
for r in zona.itertuples():
    G.add_node(int(r.id_zona), wilayah=r.wilayah)
for r in arus.itertuples(index=False):
    G.add_edge(int(r.asal), int(r.tujuan), jumlah=int(r.jumlah))
print("node:", G.number_of_nodes(), "| edge:", G.number_of_edges())

# Mudah: k langkah cukup mengganti angka cutoff
H = nx.DiGraph([(a, b) for a, b, d in G.edges(data=True) if d["jumlah"] >= MIN])
for k in (1, 2, 3):
    print(f"dalam <= {k} langkah dari zona {ZONA_AWAL}: {len(nx.single_source_shortest_path_length(H, ZONA_AWAL, cutoff=k)) - 1} zona")

# Silang dengan SQL: tepat 3 langkah = tetangga dari tetangga dari tetangga
tiga_graph = {c for a in H[ZONA_AWAL] for b in H[a] for c in H[b]}
print("tepat 3 langkah (graph):", len(tiga_graph), "| sama dengan SQL?", tiga_graph == tiga_sql)

# Canggung: merangkum per wilayah harus ditulis sendiri dengan loop atas semua node
per_wilayah = {}
for n, w in G.nodes(data="wilayah"):
    per_wilayah[w] = per_wilayah.get(w, 0) + G.out_degree(n, weight="jumlah")
print(sorted(per_wilayah.items(), key=lambda kv: -kv[1]))

node: 263 | edge: 44120
dalam <= 1 langkah dari zona 183: 83 zona
dalam <= 2 langkah dari zona 183: 91 zona
dalam <= 3 langkah dari zona 183: 91 zona
tepat 3 langkah (graph): 92 | sama dengan SQL? True
[('Manhattan', 391107), ('Queens', 270929), ('Brooklyn', 205487), ('Staten Island', 67041), ('Bronx', 60912), ('EWR', 4524)]


Ringkasan R-2:

| Cara menyimpan zona | Pertanyaan yang mudah | Pertanyaan yang canggung |
|---|---|---|
| Tabel (SQLite) | Berapa zona dan penjemputan di tiap wilayah, satu GROUP BY | Zona yang terjangkau dalam tiga langkah, butuh dua self-join dan bertambah satu JOIN tiap langkah |
| Document (TinyDB) | Profil lengkap satu zona beserta tiga tujuan teratasnya, satu kali baca | Zona yang paling sering menjadi tujuan teratas zona lain, semua dokumen harus dibuka |
| Graph (NetworkX) | Zona yang terjangkau dalam k langkah, cukup mengganti angka cutoff | Jumlah penjemputan per wilayah, harus loop atas semua node |

## R-3. Delta Lake Dua Hari

Hari pertama memakai perjalanan tanggal 1 Januari 2023, hari kedua tanggal 2 Januari 2023. Hari pertama saya tulis dalam dua tahap (pagi dan sore). Hari kedua saya tambah data baru lalu saya koreksi beberapa baris.

In [11]:
from deltalake import DeltaTable, write_deltalake

KOLOM_DELTA = ["waktu_jemput", "zona_jemput", "zona_tujuan",
               "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
hari1 = df.loc[df["waktu_jemput"].dt.date == pd.Timestamp("2023-01-01").date(), KOLOM_DELTA].sort_values("waktu_jemput").reset_index(drop=True)
hari2 = df.loc[df["waktu_jemput"].dt.date == pd.Timestamp("2023-01-02").date(), KOLOM_DELTA].sort_values("waktu_jemput").reset_index(drop=True)
print("baris hari 1:", len(hari1), "| baris hari 2:", len(hari2))

# PERBAIKAN: Gunakan penyimpanan lokal Colab (/content) agar operasi rename file Delta Lake diizinkan
PATH_TUGAS = "/content/trips_delta_dua_hari"
if os.path.exists(PATH_TUGAS):
    shutil.rmtree(PATH_TUGAS)

# HARI 1 pagi: perjalanan sebelum jam 12 (versi 0)
pagi = hari1[hari1["waktu_jemput"].dt.hour < 12]
write_deltalake(PATH_TUGAS, pagi, mode="overwrite")
print("hari 1 pagi  -> versi:", DeltaTable(PATH_TUGAS).version(), "| baris:", len(DeltaTable(PATH_TUGAS).to_pandas()))

# HARI 1 sore: sisa perjalanan hari itu (versi 1)
sore = hari1[hari1["waktu_jemput"].dt.hour >= 12]
write_deltalake(PATH_TUGAS, sore, mode="append")
VERSI_AKHIR_HARI1 = DeltaTable(PATH_TUGAS).version()
print("hari 1 sore  -> versi:", VERSI_AKHIR_HARI1, "| baris:", len(DeltaTable(PATH_TUGAS).to_pandas()))

baris hari 1: 32309 | baris hari 2: 32204
hari 1 pagi  -> versi: 0 | baris: 16185
hari 1 sore  -> versi: 1 | baris: 32309


In [17]:
#hari1
shutil.rmtree(PATH_TUGAS)

pagi = hari1[hari1["waktu_jemput"].dt.hour < 12].reset_index(drop=True)
write_deltalake(PATH_TUGAS, pagi, mode="overwrite")
print("hari 1 pagi  -> versi:", DeltaTable(PATH_TUGAS).version(), "| baris:", len(DeltaTable(PATH_TUGAS).to_pandas()))

sore = hari1[hari1["waktu_jemput"].dt.hour >= 12].reset_index(drop=True)
write_deltalake(PATH_TUGAS, sore, mode="append")
VERSI_AKHIR_HARI1 = DeltaTable(PATH_TUGAS).version()
print("hari 1 sore  -> versi:", VERSI_AKHIR_HARI1, "| baris:", len(DeltaTable(PATH_TUGAS).to_pandas()))
print("kolom tabel Delta :", [f.name for f in DeltaTable(PATH_TUGAS).schema().fields])

hari 1 pagi  -> versi: 0 | baris: 16185
hari 1 sore  -> versi: 1 | baris: 32309
kolom tabel Delta : ['waktu_jemput', 'zona_jemput', 'zona_tujuan', 'wilayah_jemput', 'jarak_mil', 'total_bayar', 'metode_bayar']


In [18]:
# HARI 2: menambah perjalanan tanggal 2 (versi 2)
write_deltalake(PATH_TUGAS, hari2, mode="append")
dt = DeltaTable(PATH_TUGAS)
print("hari 2 tambah  -> versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# HARI 2: koreksi. Anggap perjalanan berstatus Sengketa di Queens kelebihan tagih 3.0
semua = pd.concat([hari1, hari2])
n_koreksi = int(((semua["metode_bayar"] == "Sengketa") & (semua["wilayah_jemput"] == "Queens")).sum())
print("baris yang akan dikoreksi:", n_koreksi)
dt.update(predicate="metode_bayar = 'Sengketa' AND wilayah_jemput = 'Queens'",
          updates={"total_bayar": "total_bayar - 3.0"})
dt = DeltaTable(PATH_TUGAS)
print("hari 2 koreksi -> versi:", dt.version(), "| baris:", len(dt.to_pandas()))

hari 2 tambah  -> versi: 2 | baris: 64513
baris yang akan dikoreksi: 203
hari 2 koreksi -> versi: 3 | baris: 64513


In [19]:
# Membaca tabel seperti keadaannya di akhir hari pertama
akhir_hari1 = DeltaTable(PATH_TUGAS, version=VERSI_AKHIR_HARI1).to_pandas()
terbaru = DeltaTable(PATH_TUGAS).to_pandas()

print("baris akhir hari 1:", len(akhir_hari1), "| baris terbaru:", len(terbaru))
print("waktu_jemput terakhir pada versi akhir hari 1:", akhir_hari1["waktu_jemput"].max())
print("jumlah baris sama dengan data hari 1?", len(akhir_hari1) == len(hari1))
print("total_bayar akhir hari 1:", round(akhir_hari1["total_bayar"].sum(), 2), "| data hari 1 asli:", round(hari1["total_bayar"].sum(), 2))

# Koreksi hari 2 ikut mengubah baris hari 1 di versi terbaru, tetapi versi akhir hari 1 tetap utuh
h1_terbaru = terbaru[terbaru["waktu_jemput"] < "2023-01-02"]
n_koreksi_h1 = int(((hari1["metode_bayar"] == "Sengketa") & (hari1["wilayah_jemput"] == "Queens")).sum())
print("selisih total_bayar baris hari 1 (akhir hari 1 - terbaru):",
      round(akhir_hari1["total_bayar"].sum() - h1_terbaru["total_bayar"].sum(), 2),
      "| baris hari 1 yang dikoreksi:", n_koreksi_h1, "x 3.0 =", n_koreksi_h1 * 3.0)

baris akhir hari 1: 32309 | baris terbaru: 64513
waktu_jemput terakhir pada versi akhir hari 1: 2023-01-01 23:59:00
jumlah baris sama dengan data hari 1? True
total_bayar akhir hari 1: 590498.01 | data hari 1 asli: 590498.01
selisih total_bayar baris hari 1 (akhir hari 1 - terbaru): 333.0 | baris hari 1 yang dikoreksi: 111 x 3.0 = 333.0


In [20]:
# Riwayat tabel
for h in DeltaTable(PATH_TUGAS).history():
    print(h.get("version"), h.get("operation"), pd.to_datetime(h.get("timestamp"), unit="ms"))

# Isi _delta_log/: satu file JSON per versi, tiap baris di dalamnya adalah satu aksi
log_dir = f"{PATH_TUGAS}/_delta_log"
print(sorted(os.listdir(log_dir)))
for nama in sorted(f for f in os.listdir(log_dir) if f.endswith(".json")):
    print("\n==", nama)
    for baris in open(f"{log_dir}/{nama}"):
        aksi = json.loads(baris)
        jenis = list(aksi)[0]
        isi = aksi[jenis]
        if jenis == "commitInfo":
            print("  commitInfo | operation:", isi.get("operation"), "| parameter:", isi.get("operationParameters"))
        elif jenis in ("add", "remove"):
            print(f"  {jenis:10} | {isi['path']} | {isi.get('size', '-')} byte")
        else:
            print(f"  {jenis:10} | {json.dumps(isi)[:110]}")

3 UPDATE 2026-10-08 02:51:37.721000
2 WRITE 2026-10-08 02:51:37.638000
1 WRITE 2026-10-08 02:51:15.474000
0 WRITE 2026-10-08 02:51:15.439000
['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json', '00000000000000000003.json']

== 00000000000000000000.json
  commitInfo | operation: WRITE | parameter: {'mode': 'Overwrite'}
  protocol   | {"minReaderVersion": 3, "minWriterVersion": 7, "readerFeatures": ["timestampNtz"], "writerFeatures": ["timesta
  metaData   | {"id": "a4f4ab93-3961-49cb-afec-a15bfd2b01c0", "name": null, "description": null, "format": {"provider": "parq
  add        | part-00000-18793951-e73e-4978-9449-faa00eaf7fea-c000.snappy.parquet | 121474 byte

== 00000000000000000001.json
  commitInfo | operation: WRITE | parameter: {'mode': 'Append'}
  add        | part-00000-8aa15fb5-85fe-48ac-b7bf-c7e963ac37be-c000.snappy.parquet | 121288 byte

== 00000000000000000002.json
  commitInfo | operation: WRITE | parameter: {'mode': 'Append'}
  add      

Versi 1 adalah keadaan tabel di akhir hari pertama. Jumlah barisnya sama dengan data tanggal 1 Januari, waktu jemput terakhirnya masih 23:59 tanggal 1, dan total bayarnya sama persis dengan data hari pertama sebelum koreksi. Koreksi di hari kedua tercatat sebagai versi 3 lewat aksi `remove` pada file lama dan `add` pada file baru, sehingga file milik versi 1 tetap ada di disk dan tetap bisa dibaca.